# 🏦 Central Bank Agentic Complaint Management System
### LangGraph + LLM Classification + RAG (Regulatory Circulars)

This notebook implements:
- LLM-based issue classification
- LangGraph stateful workflow
- RAG using regulatory circulars
- Full audit trail (dates, department, resolution)


In [ ]:
!pip install -U langgraph langchain langchain-community transformers sentence-transformers faiss-cpu pydantic


In [ ]:
from typing import TypedDict
from datetime import datetime
import uuid

from langgraph.graph import StateGraph, END
from langchain_community.llms import HuggingFacePipeline
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline


In [ ]:
class ComplaintState(TypedDict):
    bank_id: str
    complaint_id: str
    complaint_text: str
    issue_category: str
    assigned_department: str
    status: str
    date_filed: str
    date_resolved: str
    resolution_summary: str
    regulatory_context: str


In [ ]:
def today():
    return datetime.now().strftime('%Y-%m-%d')

def generate_id():
    return str(uuid.uuid4())[:8]


In [ ]:
classifier_pipe = pipeline(
    'text-generation',
    model='google/flan-t5-base',
    max_new_tokens=64
)

llm = HuggingFacePipeline(pipeline=classifier_pipe)


In [ ]:
regulatory_docs = [
    'All banks must report cybersecurity incidents within 6 hours.',
    'Fraud above threshold requires immediate central bank notification.',
    'Payment failures impacting customers must be resolved within T+1.',
    'Compliance deviations must be reviewed by supervision department.'
]

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_texts(regulatory_docs, embeddings)
retriever = vectorstore.as_retriever()


In [ ]:
def intake_node(state):
    state['complaint_id'] = generate_id()
    state['date_filed'] = today()
    state['status'] = 'Received'
    return state


In [ ]:
def classify_issue(state):
    prompt = f"Classify the banking issue into Fraud, Payments, Cybersecurity, Compliance, or Supervision: {state['complaint_text']}"
    state['issue_category'] = llm(prompt).strip()
    state['status'] = 'Classified'
    return state


In [ ]:
def rag_node(state):
    docs = retriever.get_relevant_documents(state['complaint_text'])
    state['regulatory_context'] = docs[0].page_content if docs else ''
    return state


In [ ]:
def route_department(state):
    mapping = {
        'Fraud': 'Risk & Fraud Department',
        'Payments': 'Payments & Settlement Department',
        'Cybersecurity': 'Cyber Risk Department',
        'Compliance': 'Compliance Department'
    }
    state['assigned_department'] = mapping.get(state['issue_category'], 'Supervision Department')
    state['status'] = 'Assigned'
    return state


In [ ]:
def resolve_issue(state):
    state['resolution_summary'] = f"Resolved by {state['assigned_department']} using regulation: {state['regulatory_context']}"
    state['date_resolved'] = today()
    state['status'] = 'Resolved'
    return state


In [ ]:
graph = StateGraph(ComplaintState)

graph.add_node('intake', intake_node)
graph.add_node('classify', classify_issue)
graph.add_node('rag', rag_node)
graph.add_node('route', route_department)
graph.add_node('resolve', resolve_issue)

graph.set_entry_point('intake')
graph.add_edge('intake', 'classify')
graph.add_edge('classify', 'rag')
graph.add_edge('rag', 'route')
graph.add_edge('route', 'resolve')
graph.add_edge('resolve', END)

app = graph.compile()


In [ ]:
input_state = {
    'bank_id': 'RB-12',
    'complaint_text': 'Multiple failed payments detected impacting customers',
    'issue_category': '',
    'assigned_department': '',
    'status': '',
    'complaint_id': '',
    'date_filed': '',
    'date_resolved': '',
    'resolution_summary': '',
    'regulatory_context': ''
}

app.invoke(input_state)
